In [1]:
# import base64

# file = open("Graph.png", "rb")
# image = file.read()
# widgets.Image(
#     value=image,
#     format='png',
#     width=300,
#     height=400,
# )

In [2]:
from graphlib import TopologicalSorter

graph = {"D": {"B", "C"}, "C": {"A"}, "B": {"A"}}
ts = TopologicalSorter(graph)
tuple(ts.static_order())

('A', 'C', 'B', 'D')

In [4]:
! dot -c

In [31]:
from collections import OrderedDict
from graphviz import Digraph 

import ipywidgets as widgets
# from ipywidgets import interact, interactive, fixed, interact_manual
import base64
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objs as go 
import re
import urllib.parse

# instantiating object 
dot = Digraph() 

node_kwargs = dict(
    width = '1.5',
    labelloc = "t", 
    shape = 'circle', 
    # fontname = 'Arial',
    # fontname = "Comic Sans MS",
    fontname = None,
    fontsize = '25px',
)

# label_templ = '< <B>{}</B> >'
label_templ = '{}'

# Adding nodes 
# dot.node('A', 'I', pos = "0!,1!") 
# dot.node('B', 'H1', pos = "2!,2!") 
# dot.node('C', 'H2', pos = "2!,0!") 
# dot.node('D', 'O', pos = "3!,0!") 
graph = {"I": {"A", "B", "C"}, "A": {"O"}, "B": {"O"}, "C": {"O"}}
nodes = set(graph.keys())
for node, children in graph.items():
    nodes = nodes.union(children)
edges = []
for node in nodes:
    dot.node(node, label=label_templ.format(node), **node_kwargs) 
    if node in graph:
        edges.extend([node+child for child in graph[node]])
dot.edges(edges)
    
# dot.node('A', label=label_templ.format('I'), **node_kwargs) 
# dot.node('B', label=label_templ.format('A'), **node_kwargs)
# dot.node('X', label=label_templ.format('B'), **node_kwargs)
# dot.node('C', label=label_templ.format('C'), **node_kwargs) 
# dot.node('D', label=label_templ.format('O'), **node_kwargs) 
# dot.rankdir = 'LR'
# dot.edges(['AB', 'AC', 'BD', 'CD', 
#            'AX', 'XD'
#           ]) 
# dot.edge('B', 'C', constraint = 'false') 
# dot.edge('C', 'D', constraint = 'false') 

dot.graph_attr['rankdir'] = 'LR'
dot.graph_attr['size'] = "4!"
dot.graph_attr['margin'] = "0!"
dot.graph_attr['nodesep'] = "1.0"
dot.graph_attr['ranksep'] = "1.0"
# dot.graph_attr['rankjustify'] = "l"


# # saving source code 
# dot.format = 'png'
# dot.render('Graph') 
# with open("Graph.png", "rb") as image_file:
#     encoded_string = base64.b64encode(image_file.read()).decode()
# data_url = f'data:image/png;base64,{encoded_string}'

# dot.format = 'svg'
# dot.render('Graph') 
# with open("Graph.svg", "r") as image_file:
#     svg_string = image_file.read()
#     svg_string = re.search(r'<svg.*svg>', svg_string, re.DOTALL).group(0)
#     from urllib.parse import unquote
#     svg_string = unquote(svg_string)

dot.format = 'svg'
dot.render('Graph') 
with open("Graph.svg", "r") as image_file:
    svg_string = image_file.read()
    svg_string = re.search(r'<svg.*svg>', svg_string, re.DOTALL).group(0)
    svg_string = svg_string.replace('\n', ' ')
    svg_string = re.sub(r'<!--.*? -->', ' ', svg_string)
    svg_string = urllib.parse.quote(svg_string, safe='"/=-,:;()!& ').replace('"', "'")
data_url = f'data:image/svg+xml,{svg_string}'

style = widgets.HTML("""
<style>
.my-sliders{
    background-image: url("DATAURL");
    # background-size: contain;
    background-position:center;
    background-repeat: no-repeat;
    # height: 4in;
    # width: 4in;
    min-height: 4in;
    min-width: 4in;
    max-height: 4in;
    max-width: 4in;    
}
.readout-label{
  position: absolute;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
  min-width: 50px;
  max-width: 50px;  
}
.my-sliders .widget-slider { position: absolute; }
.my-sliders > :nth-child(1) { left: 0in; top: 1.9in; }
.my-sliders > :nth-child(2) { left: 0in; top: 2.3in; }
.my-sliders > :nth-child(3) { left: 1.5in; top: 1.5in; }
.my-sliders > :nth-child(4) { left: 1.5in; top: 1.7in; }
.my-sliders > :nth-child(5) { left: 3in; top: 1.5in; }
.my-sliders > :nth-child(6) { left: 3in; top: 1.7in; }

</style>
""".replace('DATAURL', data_url))

slider_layout = widgets.Layout(width='180px')
sliders = OrderedDict(
    a=widgets.FloatSlider(min=1, max=100, step=0.1,
                                  value=-10.25, continuous_update=True, description=' ', orientation='horizontal',
                                  layout=slider_layout, 
                          #readout=False, 
                          readout_format=".1f", 
                          style={"description_width": "0px", "handle_color": '#1876D2'}),
    b=widgets.FloatSlider(min=1000, max=10000, step=0.01,
                                  value=-10.25, ccontinuous_update=True, description=' ', orientation='horizontal',
                                  layout=slider_layout, readout=False, #readout_format=".1f", 
                          style={"description_width": "0px", "handle_color": '#549923'}),
    c=widgets.FloatSlider(min=-10000000, max=10000000, step=100000,
                                  value=-10.25, ccontinuous_update=True, description=' ', orientation='horizontal',
                                  layout=slider_layout, readout=False, #readout_format=".1f", 
                          style={"description_width": "0px", "handle_color": '#D9D9D9'}), #D9D9D9
)

readout = widgets.widgets.Output(value='1')               
readout.add_class("readout-label")

@readout.capture(clear_output=True)
def round_value(change):
    print(round(change.new, 2))

sliders['a'].observe(round_value, names="value")
box_list = [sliders['a'], sliders['b'], sliders['c'], readout]


play = widgets.Play(
    interval=0.01,
    value=1000,
    min=0,
    max=10000,
    step=1,
    description="Press play",
    disabled=False
)
widgets.jslink((play, 'value'), (sliders['a'], 'value'))

# box_list = []
# for s in sliders.values():
#     readout_label = widgets.Label()
#     readout_label.add_class("readout-label")
#     widgets.jsdlink((s, "value"), (readout_label, "value"))
#     # box_list.append(widgets.VBox([s, widgets.HBox([readout_label], _dom_classes=["my-readouts"])]))
#     box_list.extend([s, readout_label])
    
box = widgets.Box(
    box_list,
                   _dom_classes=["my-sliders"],
                  layout=widgets.Layout(margin='0px', 
                                        width='4in', min_width='4in', max_width='4in',
                                        height='4in', min_height='4in', max_height='4in'))

plot_output = widgets.Output(layout=widgets.Layout(margin='0px', 
                                        width='4in', min_width='4in', max_width='4in',
                                        height='4in', min_height='4in', max_height='4in'))
#layout = go.Layout(yaxis=dict(range=[0, 0.4]))

def f(a=None, b=None, c=None):
    x = np.linspace(-1200, 1200, 1000)
    with plot_output:
        plot_output.clear_output(wait=True)
        fig = go.Figure(data=go.Scatter(x=x, y=sliders['a'].value*x**2 + sliders['b'].value*x + sliders['c'].value, mode='lines'), 
                        layout_yaxis_range=[0, 1e7], 
                        layout_xaxis_range=[-1000, 1000],
                        layout_margin=dict(l=20, r=20, t=20, b=20)
                       )
        fig.update_layout(
            autosize=False,
            width=384,
            height=384,
            template='simple_white'
        )        
        display(fig)

for key in sliders:
    sliders[key].observe(f, names='value')

w = widgets.interactive(f, **sliders);

display(style, widgets.HBox([box, plot_output, play]))

HTML(value='\n<style>\n.my-sliders{\n    background-image: url("data:image/svg+xml,%3Csvg width=\'288pt\' heig…

# TODO 
- Make all the widgets for hte exercise and display them in separate cells. Thay way they "stay alive"

In [7]:
display(style, widgets.HBox([plot_output]))


HTML(value='\n<style>\n.my-sliders{\n    background-image: url("data:image/svg+xml,%3Csvg width=\'288pt\' heig…

In [8]:
display(style, widgets.HBox([box]))


HTML(value='\n<style>\n.my-sliders{\n    background-image: url("data:image/svg+xml,%3Csvg width=\'288pt\' heig…